# Neurological MRI XAI Pipeline — Kaggle Runner

Master orchestrator for training Swin + LoRA, evaluation, XAI, and Florence-2 reporting on **Kaggle Notebooks**.

**Before running:**
1. Enable a **GPU** accelerator (T4 recommended).
2. **Add Input** → [neurological-disorders-mri-dataset-for-xai](https://www.kaggle.com/datasets/engrsakib02/neurological-disorders-mri-dataset-for-xai).
3. Run cells in order.

In [ ]:
# Cell 1: Environment check
import sys

import torch

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2: Clone repository
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/engrsakib/Neurological-MRI-XAI-Pipeline.git"
PROJECT_DIR = Path("/kaggle/working/Neurological-MRI-XAI-Pipeline")

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    print(f"Repo already exists at {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Cell 3: Install dependencies
import subprocess
import sys

import torch

print(f"Kaggle torch version: {torch.__version__}")

subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run(["pip", "install", "-q", "-e", "."], check=True)
subprocess.run(
    ["pip", "install", "-q", "git+https://github.com/facebookresearch/segment-anything.git"],
    check=True,
)

sys.path.insert(0, str(PROJECT_DIR / "src"))
print("Dependencies installed.")

In [ ]:
# Cell 4: Kaggle dataset path + environment
import os
from pathlib import Path

from neuro_mri_xai.config import load_config

DATA_DIR = "/kaggle/input/datasets/engrsakib02/neurological-disorders-mri-dataset-for-xai/data"

print("Kaggle inputs:", os.listdir("/kaggle/input"))
data_path = Path(DATA_DIR)
assert data_path.exists() or data_path.parent.exists(), (
    "Attach engrsakib02/neurological-disorders-mri-dataset-for-xai via Add Input"
)

os.environ["NEURO_MRI_PROJECT_ROOT"] = str(PROJECT_DIR)
os.environ["NEURO_MRI_SAM_ENABLED"] = "false"
os.environ["NEURO_MRI_SEQUENTIAL_VRAM"] = "true"

cfg = load_config("configs/default.yaml", data_dir=DATA_DIR)
DATA_DIR = str(cfg.dataset.data_dir)
print(f"Resolved data dir: {DATA_DIR}")
print(f"Backbone: {cfg.model.backbone}, LoRA: {cfg.model.use_lora}, SAM: {cfg.sam.enabled}")

In [ ]:
# Cell 5: Download SAM weights
import subprocess

subprocess.run(["python", "scripts/download_weights.py", "--config", "configs/default.yaml"], check=True)

In [ ]:
# Cell 6: Train Swin + LoRA
import subprocess

subprocess.run(
    [
        "python",
        "-m",
        "neuro_mri_xai.training.train_cli",
        "--config",
        "configs/default.yaml",
        "--data-dir",
        DATA_DIR,
    ],
    check=True,
)

In [ ]:
# Cell 7: Evaluate on test set
import subprocess

subprocess.run(
    [
        "python",
        "-m",
        "neuro_mri_xai.evaluation.test_eval",
        "--config",
        "configs/default.yaml",
        "--checkpoint",
        "outputs/checkpoints/best_swin.pt",
        "--data-dir",
        DATA_DIR,
    ],
    check=True,
)

In [ ]:
# Cell 8: XAI visualizations (single sample)
import os
import subprocess
from pathlib import Path

os.environ["NEURO_MRI_SAM_ENABLED"] = "true"
sample_image = next(Path(DATA_DIR).rglob("*.jpg"))
print(f"Sample: {sample_image}")

subprocess.run(
    [
        "python",
        "-m",
        "neuro_mri_xai.explainability.xai_cli",
        "--config",
        "configs/default.yaml",
        "--checkpoint",
        "outputs/checkpoints/best_swin.pt",
        "--image",
        str(sample_image),
        "--data-dir",
        DATA_DIR,
        "--output-dir",
        "outputs/figures",
    ],
    check=True,
)

In [ ]:
# Cell 9: Full HTML diagnostic report
import os
from pathlib import Path

from IPython.display import HTML, display

from neuro_mri_xai.report import generate_report

os.environ["NEURO_MRI_SAM_ENABLED"] = "true"
os.environ["NEURO_MRI_SEQUENTIAL_VRAM"] = "true"

sample_image = next(Path(DATA_DIR).rglob("*.jpg"))
print(f"Sample: {sample_image}")

report_path = generate_report(
    checkpoint="outputs/checkpoints/best_swin.pt",
    image=str(sample_image),
    config_path="configs/default.yaml",
    data_dir=DATA_DIR,
)
display(HTML(report_path.read_text()))

In [ ]:
# Cell 10: Persist outputs (Kaggle saves /kaggle/working automatically)
from pathlib import Path

output_root = Path("outputs")
for folder in ["checkpoints", "figures", "reports", "logs"]:
    path = output_root / folder
    if path.exists():
        files = list(path.rglob("*"))
        print(f"{folder}: {len(files)} file(s) under {path.resolve()}")

print("\nDownload outputs from the Kaggle notebook 'Output' tab before the session ends.")